# Thesis Baseline Setup

This notebook documents the workflow for the thesis baseline on xBD/xView2 using a building-level classification pipeline.

---

## Core Principle

- Preprocessing and model training run directly in Python
- Notebook is used for:
  - Data inspection
  - Sanity checks
  - Training control
  - Evaluation
  - Visualization

The official xView2 Docker pipeline is not used as the main experimental setup, because the thesis isolates the damage classification task from localization by using ground truth building polygons.

---

## Required Data

Download and place on Desktop:

- `train`
- `test`
- `hold`

These contain:
- images
- labels

---

## Main Experimental Design

The thesis does not use the original end to end competition pipeline as the primary method.

Instead, the pipeline is:

1. Read ground truth polygons from label JSON files
2. Build a building-level metadata table
3. Extract pre/post building crops
4. Stack pre and post crops into 6-channel tensors
5. Train a ResNet50-based classifier on damage labels

This removes localization as a confounding factor and isolates the classification task.

---

## Required External Apps

- `miniconda` or another Python environment manager
- optionally `docker` only for reproducing the original released baseline, not for the main thesis experiments

---

## Create Environment

```bash
conda create -n thesis_xview python=3.10
conda activate thesis_xview

In [1]:
import sys
!{sys.executable} -m pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
  Using cached libtiff-0.4.2.tar.gz (129 kB)
     |████████████████████████████████| 948 kB 390 kB/s eta 0:00:01
     |████████████████████████████████| 323 kB 469 kB/s eta 0:00:01
     |████████████████████████████████| 1.4 MB 2.7 MB/s eta 0:00:01
  Using cached imantics-0.1.12-py3-none-any.whl
     |████████████████████████████████| 1.4 MB 1.1 MB/s eta 0:00:01
  distutils: /private/var/folders/bs/rzk6qg1902vgdlp6h22t8bq40000gn/T/pip-build-env-42t6_521/normal/lib/python3.9/site-packages
  sysconfig: /Library/Python/3.9/site-packages
  distutils: /private/var/folders/bs/rzk6qg1902vgdlp6h22t8bq40000gn/T/pip-build-env-42t6_521/normal/lib/python3.9/site-packages
  sysconfig: /Library/Python/3.9/site-packages
  user = False
  home = None
  root = None
  prefix = '/private/var/folders/bs/rzk6qg1902vgdlp6h22t8bq40000gn/T/pip-build-env-42t6_521/normal'
  distutils: /private/var/folders/bs/rzk6qg1902vgdlp6h22t8bq4000

In [3]:
import sys
!{sys.executable} -m pip install torch torchvision torchaudio

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 73.6 MB 1.2 MB/s eta 0:00:011     |████████████████████████████    | 64.6 MB 15.3 MB/s eta 0:00:01
     |████████████████████████████████| 1.9 MB 1.3 MB/s eta 0:00:01
     |████████████████████████████████| 1.9 MB 12.4 MB/s eta 0:00:01
     |████████████████████████████████| 6.3 MB 896 kB/s eta 0:00:011
     |████████████████████████████████| 134 kB 885 kB/s eta 0:00:01
     |████████████████████████████████| 200 kB 1.1 MB/s eta 0:00:01
     |████████████████████████████████| 536 kB 689 kB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [2]:
import torch
import numpy as np
import pandas as pd
import cv2
import shapely

print("All packages loaded successfully")
print("Torch version:", torch.__version__)

All packages loaded successfully
Torch version: 2.8.0


# Training, Validation, and Class Weighting Strategy

## 1. Data Splits: Train, Test, and Hold

The dataset is divided into three distinct subsets, each with a specific role in the machine learning pipeline:

### Train Split
The **training set** is used to learn the model parameters. During this phase, the model updates its weights based on the input data and corresponding labels.

### Test Split
The **test set** is used during training to evaluate model performance after each epoch. It serves as a **validation set** for:

- monitoring learning progress  
- selecting the best model (e.g., best epoch)  
- comparing configurations or hyperparameters  

Importantly, the model does **not learn from the test set**. It is only used for evaluation.

### Hold Split
The **holdout set** is used for the **final evaluation** after training is complete. It provides an unbiased estimate of model performance because:

- it is not used during training  
- it is not used for model selection  

This separation ensures that reported results reflect true generalization.

---

### Summary of Roles

| Split | Purpose |
|------|--------|
| Train | Learn model parameters |
| Test | Model selection and monitoring |
| Hold | Final unbiased evaluation |

---

## 2. Class Imbalance and Class Weights

### Problem: Imbalanced Data

In the dataset, the distribution of damage classes is highly imbalanced. For example:

- "no-damage" appears far more frequently  
- "minor", "major", and "destroyed" are much less common  

Without correction, the model would tend to predict the majority class ("no-damage") to minimize loss, leading to poor performance on rare but important damage categories.

---

### Solution: Class-Weighted Loss

To address this, a **class-weighted cross-entropy loss** is used.

Each class is assigned a weight:

```text
[0.3402, 2.6667, 2.8210, 3.0202]

In [18]:
import sys
!{sys.executable} model/classification_training.py

Loading data...

Split sizes:
split
train    159793
test      53850
hold      53137
Name: count, dtype: int64

Train label distribution:
damage_label
no-damage       117425
minor-damage     14980
major-damage     14161
destroyed        13227
Name: count, dtype: int64

Using device: mps

Using class weights: [0.34020224 2.6667724  2.8210049  3.0202048 ]

Starting training...
Epoch 1/5:  42%|██████▎        | 2084/4994 [09:40<15:18,  3.17it/s, loss=1.2738]^C


## Confusion Matrix

In [10]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import numpy as np

y_true = final_hold["targets"]
y_pred = final_hold["preds"]

labels = ["no-damage", "minor-damage", "major-damage", "destroyed"]

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(6, 5))
plt.imshow(cm_norm, interpolation='nearest')
plt.colorbar()

plt.xticks(np.arange(len(labels)), labels, rotation=45)
plt.yticks(np.arange(len(labels)), labels)

for i in range(len(labels)):
    for j in range(len(labels)):
        plt.text(j, i, f"{cm_norm[i, j]:.2f}",
                 ha="center", va="center")

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Normalized Confusion Matrix (Hold Set)")
plt.tight_layout()
plt.show()

NameError: name 'final_hold' is not defined

## Sanity Check: Verification of ResNet50-Based Architecture

Before proceeding with full training and experimentation, a structural sanity check was performed to ensure that the implemented model correctly corresponds to a ResNet50 architecture adapted to the specific task.


In [3]:
import torch
from torchvision.models import resnet50, ResNet50_Weights
import torch.nn as nn

class ResNet50SixChannel(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        weights = ResNet50_Weights.IMAGENET1K_V2
        self.backbone = resnet50(weights=weights)

        old_conv = self.backbone.conv1
        new_conv = nn.Conv2d(
            in_channels=6,
            out_channels=old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False,
        )

        with torch.no_grad():
            new_conv.weight[:, :3, :, :] = old_conv.weight
            new_conv.weight[:, 3:, :, :] = old_conv.weight

        self.backbone.conv1 = new_conv
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, num_classes)

    def forward(self, x):
        return self.backbone(x)

model = ResNet50SixChannel(num_classes=4)

print(model.backbone.conv1)
print(model.backbone.fc)

total_params = sum(p.numel() for p in model.parameters())
print("Total params:", total_params)

Conv2d(6, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
Linear(in_features=2048, out_features=4, bias=True)
Total params: 23525636




The structural sanity check confirms that the implemented model correctly corresponds to a ResNet50-based architecture adapted for the task.

### Key Results

- **Input Layer:**  
  `Conv2d(6, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))`  
  → Confirms successful adaptation from 3 to **6 input channels**, enabling joint processing of pre- and post-disaster imagery.

- **Output Layer:**  
  `Linear(in_features=2048, out_features=4)`  
  → Confirms correct modification of the classifier head for **4 damage classes**, while preserving the standard ResNet50 feature dimension.

- **Parameter Count:**  
  ~23.5 million parameters  
  → Consistent with a **ResNet50-scale model**, accounting for minor changes due to input/output layer modifications.

### Interpretation

These results verify that:

- The backbone architecture remains **ResNet50**
- The model is correctly configured for the **6-channel input representation**
- The classification head matches the **task-specific output space**

### Conclusion

The sanity check confirms that the model is structurally valid and correctly adapted, ensuring that subsequent training and evaluation are performed on a properly specified ResNet50-based architecture.
```
